In [38]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"  

In [53]:
# Helper functions
from anthropic.types import Message 

def add_user_message(messages, content):
    messages.append({
        "role": "user",
        "content": content.content if isinstance(content, Message) else content,
    })
def add_assistant_message(messages, content):
    messages.append({
        "role": "assistant",
        "content": content.content if isinstance(content, Message) else content,
    })


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    # 
    # return message.content[0].text # esto siempre asume que solo vamos a devolver un text block de claude... estamos asumiendo eso

    return message #esto refleja realidad..

'''
Extracting Text from Messages
Since you're now returning full message objects, create a helper to extract text when needed:


This function finds all text blocks in a message and joins them together, which is useful when you need to display the final response to users.
'''


def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

In [54]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [55]:
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"): # no es tan facil validar ese formato jeje
    if not date_format:
        raise ValueError("date_format cannot be empty") # al menos validamos el formato
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and time as a formatted string, using the system clock of the machine running this tool (server-local time, not necessarily the user's time zone). Use this whenever you need to know today's date, the current time, or to compute relative dates (e.g. 'tomorrow', 'in 3 days'), since you cannot know the current date from training data. The output is formatted with Python strftime directives; if date_format is omitted, the result looks like '2026-11-16 19:45:00'. Does not accept a time zone and cannot return the time for other locations.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "Optional. A Python strftime format string that controls how the date and time are rendered, e.g. '%Y-%m-%d' for '2026-11-16', '%H:%M' for '19:45', or '%A, %B %d, %Y' for 'Monday, November 16, 2026'. Must be a non-empty string containing at least one '%' directive. Defaults to '%Y-%m-%d %H:%M:%S' when omitted.",
        "default": "%Y-%m-%d %H:%M:%S",
        "minLength": 1
      }
    },
    "required": []
  }
});

In [42]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema], # json schemas que creamos
)

response

# claude no guarda el historial de mensajes
messages.append({
    "role": "assistant",
    "content": response.content
})

messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_015mMTsD1PREkdtzHsvjca7R', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}]

In [56]:
result = get_current_datetime(**response.content[0].input)

In [57]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[0].id,
        "content": result,
        "is_error": False
    }]
})


# entire conversation history.
messages

[{'role': 'user',
  'content': 'What is the current time in HH:MM format? Also, what is the current time in SS format?'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll get the current time in both formats for you.", type='text'),
   ToolUseBlock(id='toolu_013VNobF9hjm2LZqgVANKavh', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use', toolset_name=None),
   ToolUseBlock(id='toolu_01PmDatrcB5zEGUejNCht3jk', caller=DirectCaller(type='direct'), input={'date_format': '%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_013VNobF9hjm2LZqgVANKavh',
    'content': '23:25',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01PmDatrcB5zEGUejNCht3jk',
    'content': '30',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time

In [45]:
# final result...

client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

Message(id='msg_011Ch9kqsmeaCTawPpZL9QH8', container=None, content=[TextBlock(citations=None, text='The current time is **23:23:54**.', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=918, output_tokens=14, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed='standard'), diagnostics=None)

Sí, tu entendimiento es correcto. Esa es exactamente la idea del tool use en la Messages API.

Flujo mental

1. User pregunta (necesita dato “en vivo”: hora, API, DB…)
        ↓
2. Claude ve las tools en el request
        ↓
3. Claude responde con tool_use (no inventa la hora)
        ↓
4. TÚ ejecutas la función en Python (get_current_datetime)
        ↓
5. User message con tool_result (el string real)
        ↓
6. Segundo request con toda la conversación + tools
        ↓
7. Claude responde en lenguaje natural (TextBlock) usando ese resultado


In [35]:
# aca veremos el refractor

messages = []

add_user_message(messages, "Whats the current time in HH:MM:SS format?")
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

add_assistant_message(messages, response.content)

messages

[{'role': 'user', 'content': 'Whats the current time in HH:MM:SS format?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_016KV1yPEn9NwenN6fMxxftE', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}]

In [ ]:
messages = []

add_user_message(messages, "Whats the current time in HH:MM:SS format?")
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

response # nótese el stop reason tool_use, es decir que claude necesita ejecutar un tool.

Message(id='msg_011Ch9fnN5JFndWjFmUrxrCj', container=None, content=[ToolUseBlock(id='toolu_014qJDzRbUW18b1xoi3aBB4c', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=838, output_tokens=62, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed='standard'), diagnostics=None)

In [58]:
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)


def tool_result_content(tool_output):
    if isinstance(tool_output, str):
        return tool_output
    return json.dumps(tool_output)


def run_tools(message):
    tools_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    tool_result_blocks = []

    for tool_request in tools_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": tool_result_content(tool_output),
                "is_error": False,
            })
        except Exception as e:
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            })

    return tool_result_blocks

In [59]:
def run_conversation(messages, tools=None):
    if tools is None:
        tools = [get_current_datetime_schema, 
                 add_duration_to_datetime_schema,
                 set_reminder_schema]

    while True:
        response = chat(messages, tools=tools)

        add_assistant_message(messages, response)

        text = text_from_message(response)
        if text:
            print(text)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [48]:
messages = []
add_user_message(messages, "What's the current time in HH:MM:SS format?")
run_conversation(messages)

The current time is **23:24:17**.


[{'role': 'user', 'content': "What's the current time in HH:MM:SS format?"},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01DXbwJ4bMbAaRfvBA5xv6yi', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01DXbwJ4bMbAaRfvBA5xv6yi',
    'content': '23:24:17',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time is **23:24:17**.', type='text')]}]

In [49]:
messages = []
add_user_message(
    messages,
    "What is the current time in HH:MM format? Also, what is the current time in SS format?"
)
run_conversation(messages)

I'll get the current time in both formats for you.
The current time is:
- **HH:MM format**: 23:25
- **SS format**: 30 (seconds)


[{'role': 'user',
  'content': 'What is the current time in HH:MM format? Also, what is the current time in SS format?'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll get the current time in both formats for you.", type='text'),
   ToolUseBlock(id='toolu_013VNobF9hjm2LZqgVANKavh', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use', toolset_name=None),
   ToolUseBlock(id='toolu_01PmDatrcB5zEGUejNCht3jk', caller=DirectCaller(type='direct'), input={'date_format': '%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_013VNobF9hjm2LZqgVANKavh',
    'content': '23:25',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01PmDatrcB5zEGUejNCht3jk',
    'content': '30',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time

# varias tool en una misma llamada en la misma conversacion..

In [60]:
messages = []
add_user_message(
    messages,
    "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050."
)
run_conversation(messages)

I'll help you set a reminder for your doctor's appointment. Let me first calculate the exact date that's 177 days after January 1st, 2050.
Now I'll set the reminder for your doctor's appointment on June 27, 2050.
----
Setting the following reminder for 2050-06-27T09:00:00:
Doctor's appointment
----
Perfect! I've set a reminder for your doctor's appointment on **Monday, June 27, 2050 at 9:00 AM**. That's 177 days after January 1st, 2050. The reminder will notify you at that time.


[{'role': 'user',
  'content': 'Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for your doctor's appointment. Let me first calculate the exact date that's 177 days after January 1st, 2050.", type='text'),
   ToolUseBlock(id='toolu_01HerYTctXWR8bgdL8LvhLMx', caller=DirectCaller(type='direct'), input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days', 'input_format': '%Y-%m-%d'}, name='add_duration_to_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01HerYTctXWR8bgdL8LvhLMx',
    'content': 'Monday, June 27, 2050 12:00:00 AM',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set the reminder for your doctor's appointment on June 27, 2050.", type='text'),
   ToolUseBlock(id='toolu_01XxcepznKcthjptuWyJe3eZ', calle